# Simple RNN: Practice Notebook

**Task you will build:** predict the next value of a **sine wave** from the previous 5 values.

You will fill in the blanks marked `# TODO` (they are set to `None`). After each exercise there is a
**check cell** that tells you whether your answer is right.

**How to use this notebook**
1. Run the Setup cell.
2. For each exercise: read the task, fill in the `TODO`, run it, then run the check cell underneath.
3. Stuck? Open the **Hint**. Still stuck? Solutions are at the very bottom, so try on your own first.

**Reminder: how an RNN works.** It reads the sequence one step at a time and keeps a memory `h`:

`h_t = tanh( x_t · Wx + h_(t-1) · Wh + b )`

## Setup

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

np.random.seed(0)
tf.random.set_seed(0)
print("Setup done. TensorFlow version:", tf.__version__)

## Exercise 1: Create the data
Make a sine wave. `x_values` is given; set `series` to the sine of `x_values`.

<details><summary>Hint</summary>

NumPy has `np.sin(...)` which works on a whole array at once.
</details>

In [ ]:
x_values = np.linspace(0, 20, 200)     # 200 points from 0 to 20
series = None                          # TODO: apply np.sin to x_values

if series is not None:
    plt.figure(figsize=(8, 2.5)); plt.plot(series); plt.title("Sine wave"); plt.show()

In [ ]:
# CHECK 1
assert series is not None, "Fill in series (Exercise 1)"
assert series.shape == (200,), f"series should have 200 values, got {series.shape}"
assert abs(series.max() - 1) < 0.01, "Values should go up to about 1 (is it a sine wave?)"
print("Exercise 1 correct!")

## Exercise 2: Make (past 5 values → next value) examples
Slide a window of 5 along the series. Each window is an **input**; the value right after it is the **target**.

Example: `series[0:5]` is the first input and `series[5]` is its target.

<details><summary>Hint</summary>

Loop `i` over `range(len(series) - window)`, then `X.append(series[i : i + window])` and `y.append(series[i + window])`.
</details>

In [ ]:
window = 5
X, y = [], []

# TODO: write a for loop that fills X (inputs) and y (targets)


X = np.array(X)
y = np.array(y)
print("X shape:", X.shape, "| y shape:", y.shape)

In [ ]:
# CHECK 2
assert X.shape == (195, 5), f"X should be (195, 5), got {X.shape}. Did you loop over len(series) - window?"
assert y.shape == (195,), f"y should be (195,), got {y.shape}"
assert np.isclose(y[0], series[5]), "The first target should be series[5]"
assert np.allclose(X[1], series[1:6]), "The second input should be series[1:6]"
print("Exercise 2 correct!")

## Exercise 3: Reshape for the RNN
An RNN needs **3D** input: `(examples, time steps, features per step)`.
Here: 195 examples, 5 time steps, 1 feature (the value itself).

<details><summary>Hint</summary>

`X.reshape(-1, window, 1)`; the `-1` means "work out this number for me".
</details>

In [ ]:
X = None      # TODO: reshape X to (examples, window, 1)

if X is not None:
    print("X shape:", X.shape)

In [ ]:
# CHECK 3
assert X is not None, "Fill in X (Exercise 3)"
assert X.shape == (195, 5, 1), f"X should be (195, 5, 1), got {X.shape}"
print("Exercise 3 correct!")

## Exercise 4: Train / test split
Use the **first 80%** of examples for training and the **last 20%** for testing.
For time series we split **in time order**, never randomly.

<details><summary>Hint</summary>

Use slicing: `X[:split]` is the first part and `X[split:]` is the rest.
</details>

In [ ]:
split = int(len(X) * 0.8)

X_train, y_train = None, None     # TODO: first `split` examples
X_test,  y_test  = None, None     # TODO: the remaining examples

In [ ]:
# CHECK 4
assert X_train is not None and X_test is not None, "Fill in the split (Exercise 4)"
assert X_train.shape == (156, 5, 1), f"X_train should be (156, 5, 1), got {X_train.shape}"
assert X_test.shape == (39, 5, 1), f"X_test should be (39, 5, 1), got {X_test.shape}"
assert y_train.shape == (156,) and y_test.shape == (39,), "y_train / y_test have the wrong size"
print("Exercise 4 correct!")

## Exercise 5: Build the model
Build a `Sequential` model with these layers:
1. `Input(shape=(window, 1))`
2. `SimpleRNN` with **10 units** and `tanh` activation
3. `Dense` with **1 output** (the predicted next value)

<details><summary>Hint</summary>

`SimpleRNN(10, activation="tanh")` and `Dense(1)`.
</details>

In [ ]:
model = None      # TODO: Sequential([ ... three layers ... ])

if model is not None:
    model.summary()

In [ ]:
# CHECK 5
assert model is not None, "Fill in model (Exercise 5)"
assert len(model.layers) == 2, "Input is not counted, so you need 2 layers: SimpleRNN and Dense"
assert isinstance(model.layers[0], SimpleRNN), "First layer should be SimpleRNN"
assert model.layers[0].units == 10, "SimpleRNN should have 10 units"
assert isinstance(model.layers[1], Dense) and model.layers[1].units == 1, "Last layer should be Dense(1)"
print("Exercise 5 correct!  (Parameters: SimpleRNN 10 + 100 + 10 = 120, Dense 11 -> 131 total)")

## Exercise 6: Compile and train
1. Compile with optimizer `"adam"` and loss `"mse"`.
2. Train on the training data for **100 epochs** with `batch_size=16` and `verbose=0`.
3. Save the result of `fit` in a variable called `history`.

<details><summary>Hint</summary>

`model.compile(optimizer="adam", loss="mse")` then `history = model.fit(X_train, y_train, epochs=100, ...)`.
</details>

In [ ]:
# TODO: compile the model


history = None      # TODO: model.fit(...)

if history is not None:
    plt.plot(history.history["loss"]); plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Training loss"); plt.show()

In [ ]:
# CHECK 6
assert history is not None, "Fill in history (Exercise 6)"
losses = history.history["loss"]
assert len(losses) == 100, f"Train for 100 epochs, you trained {len(losses)}"
assert losses[-1] < losses[0], "Loss should go down while training"
print(f"Exercise 6 correct!  Loss went from {losses[0]:.4f} to {losses[-1]:.4f}")

## Exercise 7: Evaluate and predict
1. Get the test loss with `model.evaluate`.
2. Predict on `X_test` and flatten the result to a 1D array called `pred`.

<details><summary>Hint</summary>

`pred = model.predict(X_test, verbose=0).flatten()`
</details>

In [ ]:
test_loss = None     # TODO: model.evaluate(X_test, y_test, verbose=0)
pred = None          # TODO: model.predict(...) then .flatten()

if pred is not None:
    print("Test loss:", test_loss)
    print("Actual   Predicted")
    for a, p in list(zip(y_test, pred))[:5]:
        print(f"{a:7.3f}  {p:9.3f}")
    plt.figure(figsize=(8, 3)); plt.plot(y_test, label="actual"); plt.plot(pred, "r--", label="predicted")
    plt.legend(); plt.title("Test set: actual vs predicted"); plt.show()

In [ ]:
# CHECK 7
assert pred is not None, "Fill in pred (Exercise 7)"
assert pred.shape == y_test.shape, f"pred should have shape {y_test.shape}, got {pred.shape}. Did you use .flatten()?"
mae = np.mean(np.abs(pred - y_test))
print(f"Average error (MAE): {mae:.3f}")
print("Great, the RNN learned the wave!" if mae < 0.3 else "Error is high; try training for more epochs (Exercise 6).")

## Exercise 8: The working: do the RNN maths yourself
Redo what `SimpleRNN` does for the first test example, using the trained weights.

For each of the 5 values `x_t`:  `h = tanh( x_t @ Wx + h @ Wh + b )`

Then the Dense layer gives the prediction: `h @ Wd + bd`. It should match `pred[0]`.

<details><summary>Hint</summary>

Inside the loop: `h = np.tanh(x_t @ Wx + h @ Wh + b)`. After the loop: `manual_pred = (h @ Wd + bd).item()`.
</details>

In [ ]:
Wx, Wh, b = model.layers[0].get_weights()     # RNN weights
Wd, bd = model.layers[1].get_weights()        # Dense weights

h = np.zeros(10)               # memory starts empty
sample = X_test[0]             # shape (5, 1): five time steps

# TODO: loop over the 5 values in `sample` and update h each time


manual_pred = None             # TODO: use h with Wd and bd

if manual_pred is not None:
    print(f"Your prediction : {manual_pred:.5f}")
    print(f"TensorFlow's    : {pred[0]:.5f}")

In [ ]:
# CHECK 8
assert manual_pred is not None, "Fill in manual_pred (Exercise 8)"
assert np.isclose(manual_pred, pred[0], atol=1e-4), "Not matching yet. Check the formula and that you loop over ALL 5 steps"
print("Exercise 8 correct! You just did an RNN forward pass by hand.")

## Exercise 9 (Challenge): Dropout and Early Stopping
Build a second model, `model2`, like the first but with:
- a `Dropout(0.2)` layer between the RNN and the Dense layer (randomly switches off 20% of units while training, to reduce overfitting)
- an `EarlyStopping` callback (`monitor="val_loss"`, `patience=10`, `restore_best_weights=True`)
- up to 300 epochs and `validation_split=0.2`

Then print how many epochs it actually trained for.

<details><summary>Hint</summary>

Pass `callbacks=[early_stop]` and `validation_split=0.2` to `fit`. The number of epochs run is `len(history2.history["loss"])`.
</details>

In [ ]:
# TODO: build model2, compile it, and train it with early stopping

## Exercise 10: Experiment and think
Change one thing at a time in Exercises 2 to 6 and observe the result:
1. Make `window` **2** instead of 5. Does accuracy get worse?
2. Change the RNN from **10** to **2** units, then to **50**.
3. Train for only **5 epochs**. What happens to the predictions?

**Questions** *(answers are in the solutions section)*
- Q1. Why do we reshape the input to 3D?
- Q2. Why do we split by time instead of randomly?
- Q3. What does the `Wh` weight matrix do that a normal Dense layer does not?